# Kaggle Preprocess Export

Preprocess one source dataset shard at a time inside Kaggle, write the result to `/kaggle/working`, and then optionally publish that folder directly with the Kaggle CLI.

This notebook treats `/kaggle/working` as the conservative limit. Kaggle sessions may show more total disk than the older 20 GB autosave guidance, but only the shard written under `/kaggle/working` is meant to persist cleanly.

Recommended starting plan:
- `nabirds`: `NUM_SHARDS = 1`
- `birdsnap`: `NUM_SHARDS = 3` or `4`
- `inaturalist`: `NUM_SHARDS = 1`

Typical workflow:
1. Attach the raw source datasets.
2. Set `EXPORT_MODE`, `NUM_SHARDS`, and `ACTIVE_SHARD` in the config cell.
3. If you want automatic publishing, enable notebook internet and set the Kaggle CLI auth config.
4. Run all cells.
5. Repeat for the next shard if needed.


In [ ]:
import json
import math
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from PIL import Image, ImageFile, ImageOps
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ================================================================
# CONFIG
# ================================================================
EXPORT_MODE = "birdsnap"  # "nabirds", "birdsnap", "inaturalist"
NUM_SHARDS = 4
ACTIVE_SHARD = 0  # 0-indexed
TARGET_SIZE = 240
JPEG_QUALITY = 90
TARGET_SHARD_GB = 15.0  # keep each persisted shard comfortably below a 20 GB working/output limit
DATASET_OWNER = "rogerkutyna"
MAX_WORKERS = min(4, os.cpu_count() or 1)
OVERWRITE_OUTPUT = True
DRY_RUN = False

PUBLISH_TO_KAGGLE = False
PUBLISH_MODE = "auto"  # "auto", "create", "version"
PUBLISH_PUBLIC = False  # only used when creating a brand-new dataset
PUBLISH_MESSAGE = ""
INSTALL_KAGGLE_IF_MISSING = True  # requires notebook internet access
KAGGLE_API_TOKEN_OVERRIDE = ""  # leave blank unless you intentionally want to paste a token here
KAGGLE_API_TOKEN_SECRET = ""  # optional Kaggle notebook secret name

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_PAD_RGB = tuple(int(round(c * 255)) for c in IMAGENET_MEAN)

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

SOURCE_SLUGS = {
    "nabirds": "birdbrained-nabirds",
    "birdsnap": "birdbrained-birdsnap",
    "inaturalist": "birdbrained-inaturalist",
}
TITLE_NAMES = {
    "nabirds": "BirdBrained NABirds Preprocessed",
    "birdsnap": "BirdBrained Birdsnap Preprocessed",
    "inaturalist": "BirdBrained iNaturalist Preprocessed",
}
EXTERNAL_FOLDER_NAMES = {
    "birdsnap": "Birdsnap_Dataset",
    "inaturalist": "iNaturalist_Dataset",
}
KEYWORDS = {
    "nabirds": ("nabird",),
    "birdsnap": ("birdsnap",),
    "inaturalist": ("inaturalist", "inat"),
}
EXTERNAL_PKL_NAMES = {
    "birdsnap": "birdsnap_splits.pkl",
    "inaturalist": "inat_splits.pkl",
}

assert EXPORT_MODE in SOURCE_SLUGS
assert PUBLISH_MODE in {"auto", "create", "version"}
assert 0 <= ACTIVE_SHARD < NUM_SHARDS

OUTPUT_NAME = f"{SOURCE_SLUGS[EXPORT_MODE]}-preprocessed-s{ACTIVE_SHARD+1:02d}of{NUM_SHARDS:02d}"
DATASET_REF = f"{DATASET_OWNER}/{OUTPUT_NAME}"
OUTPUT_ROOT = WORKING_ROOT / OUTPUT_NAME


def iter_dataset_mounts():
    if not KAGGLE_INPUT.exists():
        return []

    mounts = []
    seen = set()

    def add_mount(dataset_dir):
        dataset_dir = Path(dataset_dir)
        if dataset_dir.is_dir() and dataset_dir not in seen:
            seen.add(dataset_dir)
            mounts.append(dataset_dir)

    for dataset_dir in sorted(KAGGLE_INPUT.iterdir()):
        if dataset_dir.name == "datasets":
            continue
        add_mount(dataset_dir)

    datasets_root = KAGGLE_INPUT / "datasets"
    if datasets_root.exists():
        for owner_dir in sorted(datasets_root.iterdir()):
            if not owner_dir.is_dir():
                continue
            for dataset_dir in sorted(owner_dir.iterdir()):
                add_mount(dataset_dir)

    return mounts


def show_kaggle_inputs(max_children=12):
    print("Attached Kaggle inputs:")
    mounts = iter_dataset_mounts()
    if not mounts:
        print(f"  no dataset mounts found under {KAGGLE_INPUT}")
        return
    for dataset_dir in mounts:
        print(f"  {dataset_dir}")
        children = sorted(dataset_dir.iterdir())
        for child in children[:max_children]:
            suffix = "/" if child.is_dir() else ""
            print(f"    - {child.name}{suffix}")
        if len(children) > max_children:
            print(f"    ... {len(children) - max_children} more")


def resolve_dataset_path(label, candidates, validator):
    checked = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if validator(candidate):
            print(f"{label}: {candidate}")
            return candidate
    show_kaggle_inputs()
    raise FileNotFoundError(f"Could not resolve {label}. Checked: {checked}")


def dataset_candidates(relative_path, *keywords):
    matches = []
    for dataset_dir in iter_dataset_mounts():
        name = dataset_dir.name.lower()
        if any(keyword in name for keyword in keywords):
            matches.append(dataset_dir / relative_path)
    return matches


def resolve_optional_path(label, candidates, validator):
    checked = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if validator(candidate):
            print(f"{label}: {candidate}")
            return candidate
    print(f"{label}: none found")
    return None


def is_nabirds_root(path):
    required = ["images.txt", "image_class_labels.txt", "bounding_boxes.txt", "classes.txt", "images"]
    return path.is_dir() and all((path / name).exists() for name in required)


def is_class_folder_root(path):
    return path.is_dir() and any(path.glob("class_*"))


def resolve_nabirds_root():
    slug = SOURCE_SLUGS["nabirds"]
    return resolve_dataset_path(
        "NABIRDS_ROOT",
        [
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug,
            KAGGLE_INPUT / slug,
            KAGGLE_INPUT / slug / "nabirds",
            KAGGLE_INPUT / slug / "NABirds_Dataset",
            KAGGLE_INPUT / slug / "NABirds_Dataset" / "nabirds",
            *dataset_candidates("", *KEYWORDS["nabirds"]),
            *dataset_candidates("nabirds", *KEYWORDS["nabirds"]),
            *dataset_candidates("NABirds_Dataset", *KEYWORDS["nabirds"]),
            *dataset_candidates("NABirds_Dataset/nabirds", *KEYWORDS["nabirds"]),
        ],
        is_nabirds_root,
    )


def resolve_external_root(mode):
    slug = SOURCE_SLUGS[mode]
    folder_name = EXTERNAL_FOLDER_NAMES[mode]
    return resolve_dataset_path(
        f"{mode.upper()}_ROOT",
        [
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug,
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / "images",
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / folder_name,
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / folder_name / "images",
            KAGGLE_INPUT / slug,
            KAGGLE_INPUT / slug / "images",
            KAGGLE_INPUT / slug / folder_name,
            KAGGLE_INPUT / slug / folder_name / "images",
            *dataset_candidates("", *KEYWORDS[mode]),
            *dataset_candidates("images", *KEYWORDS[mode]),
            *dataset_candidates(folder_name, *KEYWORDS[mode]),
            *dataset_candidates(f"{folder_name}/images", *KEYWORDS[mode]),
        ],
        is_class_folder_root,
    )


def resolve_external_pickle_path(mode):
    filename = EXTERNAL_PKL_NAMES[mode]
    slug = SOURCE_SLUGS[mode]
    return resolve_optional_path(
        f"{mode.upper()}_PKL",
        [
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / filename,
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / "external" / filename,
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / "artifacts" / "external" / filename,
            KAGGLE_INPUT / slug / filename,
            KAGGLE_INPUT / slug / "external" / filename,
            KAGGLE_INPUT / slug / "artifacts" / "external" / filename,
            *dataset_candidates(filename, *KEYWORDS[mode]),
            *dataset_candidates(f"external/{filename}", *KEYWORDS[mode]),
            *dataset_candidates(f"artifacts/external/{filename}", *KEYWORDS[mode]),
        ],
        lambda p: p.is_file(),
    )


def resolve_birdsnap_images_txt_path():
    slug = SOURCE_SLUGS["birdsnap"]
    return resolve_optional_path(
        "BIRDSNAP_IMAGES_TXT",
        [
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / "images.txt",
            KAGGLE_INPUT / "datasets" / DATASET_OWNER / slug / "Birdsnap_Dataset" / "images.txt",
            KAGGLE_INPUT / slug / "images.txt",
            KAGGLE_INPUT / slug / "Birdsnap_Dataset" / "images.txt",
            *dataset_candidates("images.txt", *KEYWORDS["birdsnap"]),
            *dataset_candidates("Birdsnap_Dataset/images.txt", *KEYWORDS["birdsnap"]),
        ],
        lambda p: p.is_file(),
    )


def crop_resize_pad_bbox(img, bbox, size=TARGET_SIZE, pad_rgb=IMAGENET_PAD_RGB):
    x, y, w, h = bbox
    x1 = max(0, int(x))
    y1 = max(0, int(y))
    x2 = min(img.width, int(x + w))
    y2 = min(img.height, int(y + h))
    cropped = img.crop((x1, y1, x2, y2)) if x2 > x1 and y2 > y1 else img
    scale = min(size / cropped.width, size / cropped.height)
    new_w = max(1, int(round(cropped.width * scale)))
    new_h = max(1, int(round(cropped.height * scale)))
    resized = cropped.resize((new_w, new_h), resample=Image.BILINEAR)
    pad_left = (size - new_w) // 2
    pad_top = (size - new_h) // 2
    pad_right = size - new_w - pad_left
    pad_bottom = size - new_h - pad_top
    return ImageOps.expand(
        resized,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=pad_rgb,
    )


def save_image(img, dst_path):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    ext = dst_path.suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        img.save(dst_path, quality=JPEG_QUALITY)
    else:
        img.save(dst_path)
    return dst_path.stat().st_size


def assign_shards_by_bytes(df, num_shards):
    df = df.copy().reset_index(drop=True)
    df["shard"] = -1
    shard_sizes = [0] * num_shards
    for idx in df.sort_values("bytes", ascending=False).index:
        shard_idx = min(range(num_shards), key=lambda i: shard_sizes[i])
        df.at[idx, "shard"] = shard_idx
        shard_sizes[shard_idx] += int(df.at[idx, "bytes"])
    return df, shard_sizes


def suggested_shards(total_bytes):
    target_bytes = int(TARGET_SHARD_GB * 1024**3)
    return max(1, math.ceil(total_bytes / target_bytes))


def bytes_to_gb(num_bytes):
    return num_bytes / 1024**3


def build_nabirds_manifest(root):
    images_df = pd.read_csv(root / "images.txt", sep=" ", names=["image_id", "image_rel_path"])
    labels_df = pd.read_csv(root / "image_class_labels.txt", sep=" ", names=["image_id", "class_id"])
    splits_df = pd.read_csv(root / "train_test_split_8020_all_specific.txt", sep=" ", names=["image_id", "is_train"])
    bboxes_df = pd.read_csv(root / "bounding_boxes.txt", sep=" ", names=["image_id", "x", "y", "w", "h"])
    manifest_df = images_df.merge(labels_df, on="image_id", how="left")
    manifest_df = manifest_df.merge(splits_df, on="image_id", how="left")
    manifest_df = manifest_df.merge(bboxes_df, on="image_id", how="left")
    manifest_df["src_path"] = manifest_df["image_rel_path"].map(lambda p: str(root / "images" / p))
    manifest_df["bytes"] = manifest_df["src_path"].map(lambda p: Path(p).stat().st_size)
    meta = {
        "images_df": images_df,
        "labels_df": labels_df,
        "splits_df": splits_df,
        "classes_path": root / "classes.txt",
        "hierarchy_path": root / "hierarchy.txt",
    }
    return manifest_df, meta


def normalize_external_rel_path(image_path):
    path = Path(str(image_path))
    parts = path.parts
    if "images" in parts:
        idx = parts.index("images")
        remainder = parts[idx + 1:]
        if remainder:
            return Path(*remainder)
    for idx, part in enumerate(parts):
        if part.startswith("class_"):
            return Path(*parts[idx:])
    return None


def load_birdsnap_bbox_lookup(images_txt_path):
    df = pd.read_csv(images_txt_path, sep="\t")
    required_cols = {"bb_x1", "bb_y1", "bb_x2", "bb_y2"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise RuntimeError(f"Birdsnap images.txt missing columns {sorted(missing_cols)}: {images_txt_path}")

    for col in ["bb_x1", "bb_y1", "bb_x2", "bb_y2"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    lookup = {}
    for stream_idx, row in df.iterrows():
        coords = [row["bb_x1"], row["bb_y1"], row["bb_x2"], row["bb_y2"]]
        if any(pd.isna(value) for value in coords):
            continue
        x1, y1, x2, y2 = [float(value) for value in coords]
        w = x2 - x1
        h = y2 - y1
        if w <= 0 or h <= 0:
            continue
        lookup[int(stream_idx)] = (x1, y1, w, h)

    print(f"Loaded Birdsnap bbox metadata from images.txt: {images_txt_path} ({len(lookup):,} usable rows)")
    return lookup


def birdsnap_bbox_from_saved_path(img_path, bbox_lookup):
    match = re.match(r"train_(\d+)$", Path(img_path).stem)
    if not match:
        return None
    return bbox_lookup.get(int(match.group(1)))


def build_external_manifest(root, pkl_path=None, birdsnap_images_txt_path=None, mode=None):
    rows = []
    image_suffixes = {".jpg", ".jpeg", ".png"}

    if mode == "birdsnap" and birdsnap_images_txt_path is not None and "preprocessed" not in str(root).lower():
        bbox_lookup = load_birdsnap_bbox_lookup(birdsnap_images_txt_path)
        source_counts = {"images_txt": 0, "full_image": 0}

        for class_dir in sorted(root.glob("class_*")):
            suffix = class_dir.name.split("_", 1)[1] if "_" in class_dir.name else ""
            target = int(suffix) if suffix.isdigit() else None
            for img_path in sorted(class_dir.iterdir()):
                if not img_path.is_file() or img_path.suffix.lower() not in image_suffixes:
                    continue
                bbox = birdsnap_bbox_from_saved_path(img_path, bbox_lookup)
                if bbox is None:
                    x = y = w = h = None
                    bbox_source = "full_image"
                else:
                    x, y, w, h = bbox
                    bbox_source = "images_txt"
                source_counts[bbox_source] += 1
                rows.append({
                    "rel_path": str(img_path.relative_to(root)),
                    "class_dir": class_dir.name,
                    "src_path": str(img_path),
                    "bytes": img_path.stat().st_size,
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "target": target,
                    "bbox_source": bbox_source,
                })

        if not rows:
            raise RuntimeError(f"No images found under {root}")

        print(f"  Birdsnap bbox sources: {source_counts}")
        return pd.DataFrame(rows)

    if pkl_path is not None:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
        train_df = data.get("train_df")
        if train_df is None:
            raise RuntimeError(f"External pickle missing train_df: {pkl_path}")

        required_cols = {"image_path", "x", "y", "w", "h", "target"}
        missing_cols = required_cols - set(train_df.columns)
        if missing_cols:
            raise RuntimeError(f"External pickle missing columns {missing_cols}: {pkl_path}")

        missing_paths = 0
        for row in train_df.itertuples(index=False):
            rel_path = normalize_external_rel_path(row.image_path)
            if rel_path is None:
                missing_paths += 1
                continue
            src_path = root / rel_path
            if not src_path.exists():
                missing_paths += 1
                continue
            class_dir = rel_path.parts[0] if rel_path.parts else ""
            rows.append({
                "rel_path": str(rel_path),
                "class_dir": class_dir,
                "src_path": str(src_path),
                "bytes": src_path.stat().st_size,
                "x": float(row.x),
                "y": float(row.y),
                "w": float(row.w),
                "h": float(row.h),
                "target": int(row.target),
                "bbox_source": "pickle",
            })

        if rows:
            print(f"Loaded external bbox metadata from pickle: {pkl_path}")
            if missing_paths:
                print(f"  Warning: skipped {missing_paths:,} rows whose paths did not map into {root}")
            return pd.DataFrame(rows)

        print(f"Warning: external pickle {pkl_path} did not map any usable rows into {root}; falling back to full-image export")

    for class_dir in sorted(root.glob("class_*")):
        suffix = class_dir.name.split("_", 1)[1] if "_" in class_dir.name else ""
        target = int(suffix) if suffix.isdigit() else None
        for img_path in sorted(class_dir.iterdir()):
            if img_path.is_file() and img_path.suffix.lower() in image_suffixes:
                rows.append({
                    "rel_path": str(img_path.relative_to(root)),
                    "class_dir": class_dir.name,
                    "src_path": str(img_path),
                    "bytes": img_path.stat().st_size,
                    "x": None,
                    "y": None,
                    "w": None,
                    "h": None,
                    "target": target,
                    "bbox_source": "full_image",
                })
    if not rows:
        raise RuntimeError(f"No images found under {root}")
    return pd.DataFrame(rows)


print(f"EXPORT_MODE={EXPORT_MODE} | NUM_SHARDS={NUM_SHARDS} | ACTIVE_SHARD={ACTIVE_SHARD}")
print(f"MAX_WORKERS={MAX_WORKERS} | TARGET_SIZE={TARGET_SIZE} | TARGET_SHARD_GB={TARGET_SHARD_GB}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")
print(f"DATASET_REF={DATASET_REF} | PUBLISH_TO_KAGGLE={PUBLISH_TO_KAGGLE} | PUBLISH_MODE={PUBLISH_MODE}")


In [ ]:
EXTERNAL_PKL_PATH = None
BIRDSNAP_IMAGES_TXT_PATH = None
if EXPORT_MODE == "nabirds":
    SOURCE_ROOT = resolve_nabirds_root()
    manifest_df, meta = build_nabirds_manifest(SOURCE_ROOT)
else:
    SOURCE_ROOT = resolve_external_root(EXPORT_MODE)
    EXTERNAL_PKL_PATH = resolve_external_pickle_path(EXPORT_MODE)
    if EXPORT_MODE == "birdsnap":
        BIRDSNAP_IMAGES_TXT_PATH = resolve_birdsnap_images_txt_path()
    manifest_df = build_external_manifest(
        SOURCE_ROOT,
        EXTERNAL_PKL_PATH,
        BIRDSNAP_IMAGES_TXT_PATH,
        EXPORT_MODE,
    )
    meta = None

manifest_df, shard_sizes = assign_shards_by_bytes(manifest_df, NUM_SHARDS)
active_df = manifest_df[manifest_df["shard"] == ACTIVE_SHARD].copy().reset_index(drop=True)

total_raw_bytes = int(manifest_df["bytes"].sum())
active_raw_bytes = int(active_df["bytes"].sum())
recommended_shards = suggested_shards(total_raw_bytes)

print(f"Resolved source root: {SOURCE_ROOT}")
if EXTERNAL_PKL_PATH is not None:
    print(f"Resolved external metadata pickle: {EXTERNAL_PKL_PATH}")
if BIRDSNAP_IMAGES_TXT_PATH is not None:
    print(f"Resolved Birdsnap images.txt: {BIRDSNAP_IMAGES_TXT_PATH}")
print(f"Total items: {len(manifest_df):,}")
print(f"Total raw input size: {bytes_to_gb(total_raw_bytes):.2f} GB")
print(f"Recommended minimum shards at {TARGET_SHARD_GB:.1f} GB target: {recommended_shards}")
if EXPORT_MODE != "nabirds" and "bbox_source" in manifest_df.columns:
    bbox_rows = int(manifest_df[["x", "y", "w", "h"]].notna().all(axis=1).sum())
    print(f"External bbox rows: {bbox_rows:,}/{len(manifest_df):,} | sources: {manifest_df['bbox_source'].value_counts().to_dict()}")
print()
print("Shard estimates (based on raw input bytes):")
for shard_idx, shard_bytes in enumerate(shard_sizes):
    marker = " <-- active" if shard_idx == ACTIVE_SHARD else ""
    print(f"  shard {shard_idx}: {bytes_to_gb(shard_bytes):.2f} GB{marker}")
print()
print(f"Active shard item count: {len(active_df):,}")
print(f"Active shard raw input size: {bytes_to_gb(active_raw_bytes):.2f} GB")
if len(active_df) == 0:
    raise RuntimeError("Active shard is empty; check NUM_SHARDS and ACTIVE_SHARD")
active_df.head()


In [ ]:
def write_dataset_metadata(output_root):
    shard_label = f"S{ACTIVE_SHARD+1:02d}of{NUM_SHARDS:02d}"
    description = (
        f"Preprocessed {EXPORT_MODE} shard {ACTIVE_SHARD+1} of {NUM_SHARDS}. "
        f"Images are resized to {TARGET_SIZE}px. "
        "NABirds exports include rewritten metadata for the preprocessed images."
    )
    metadata = {
        "title": f"{TITLE_NAMES[EXPORT_MODE]} {shard_label}",
        "subtitle": f"Preprocessed {EXPORT_MODE} shard {ACTIVE_SHARD+1} of {NUM_SHARDS} for BirdBrained training",
        "description": description,
        "id": DATASET_REF,
        "licenses": [{"name": "CC0-1.0"}],
        "keywords": ["birds", "preprocessed", EXPORT_MODE, "computer vision", "classification"],
    }
    (output_root / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def write_readme(output_root, written_bytes):
    lines = [
        f"Export mode: {EXPORT_MODE}",
        f"Source root: {SOURCE_ROOT}",
        f"Shard: {ACTIVE_SHARD} of {NUM_SHARDS}",
        f"Target size: {TARGET_SIZE}",
        f"Items: {len(active_df)}",
        f"Raw input GB: {bytes_to_gb(active_raw_bytes):.2f}",
        f"Written output GB: {bytes_to_gb(written_bytes):.2f}",
        f"Dataset ref: {DATASET_REF}",
        f"Publish requested: {PUBLISH_TO_KAGGLE}",
        "",
        "Notes:",
        "- NABirds bounding boxes are rewritten to cover the full preprocessed image.",
        "- If PUBLISH_TO_KAGGLE is enabled, notebook internet access must be turned on.",
        "- Kaggle CLI auth can come from KAGGLE_API_TOKEN, ~/.kaggle/access_token, or ~/.kaggle/kaggle.json.",
    ]
    (output_root / "README.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_nabirds_metadata(output_root):
    image_ids = set(active_df["image_id"].tolist())

    images_out = meta["images_df"][meta["images_df"]["image_id"].isin(image_ids)].copy()
    labels_out = meta["labels_df"][meta["labels_df"]["image_id"].isin(image_ids)].copy()
    splits_out = meta["splits_df"][meta["splits_df"]["image_id"].isin(image_ids)].copy()
    bbox_out = images_out[["image_id"]].copy()
    bbox_out["x"] = 0
    bbox_out["y"] = 0
    bbox_out["w"] = TARGET_SIZE
    bbox_out["h"] = TARGET_SIZE

    images_out.to_csv(output_root / "images.txt", sep=" ", header=False, index=False)
    labels_out.to_csv(output_root / "image_class_labels.txt", sep=" ", header=False, index=False)
    splits_out.to_csv(output_root / "train_test_split_8020_all_specific.txt", sep=" ", header=False, index=False)
    bbox_out.to_csv(output_root / "bounding_boxes.txt", sep=" ", header=False, index=False)

    if meta["classes_path"].exists():
        shutil.copy2(meta["classes_path"], output_root / "classes.txt")
    if meta["hierarchy_path"].exists():
        shutil.copy2(meta["hierarchy_path"], output_root / "hierarchy.txt")


def preprocess_one(task):
    src_path = Path(task["src_path"])
    dst_path = Path(task["dst_path"])
    bbox = task["bbox"]

    with Image.open(src_path) as img:
        img = img.convert("RGB")
        if bbox is None:
            bbox = (0, 0, img.width, img.height)
        img = crop_resize_pad_bbox(img, bbox, size=TARGET_SIZE, pad_rgb=IMAGENET_PAD_RGB)
        return save_image(img, dst_path)


def task_iter():
    if EXPORT_MODE == "nabirds":
        for row in active_df.itertuples(index=False):
            yield {
                "src_path": row.src_path,
                "dst_path": OUTPUT_ROOT / "images" / row.image_rel_path,
                "bbox": (row.x, row.y, row.w, row.h),
            }
    else:
        for row in active_df.itertuples(index=False):
            bbox = None
            if pd.notna(row.x) and pd.notna(row.y) and pd.notna(row.w) and pd.notna(row.h):
                bbox = (row.x, row.y, row.w, row.h)
            yield {
                "src_path": row.src_path,
                "dst_path": OUTPUT_ROOT / "images" / row.rel_path,
                "bbox": bbox,
            }


def collect_output_stats(output_root):
    image_suffixes = {".jpg", ".jpeg", ".png"}
    total_file_bytes = 0
    image_file_bytes = 0
    image_file_count = 0

    for path in output_root.rglob("*"):
        if not path.is_file():
            continue
        size = path.stat().st_size
        total_file_bytes += size
        if path.suffix.lower() in image_suffixes:
            image_file_bytes += size
            image_file_count += 1

    avg_image_kb = (image_file_bytes / image_file_count / 1024) if image_file_count else 0.0
    return {
        "total_file_bytes": int(total_file_bytes),
        "image_file_bytes": int(image_file_bytes),
        "image_file_count": int(image_file_count),
        "avg_image_kb": avg_image_kb,
    }


def run_command(cmd, env=None, check=True):
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result


def ensure_kaggle_cli():
    probe = subprocess.run(
        [sys.executable, "-m", "kaggle", "--help"],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        return

    if not INSTALL_KAGGLE_IF_MISSING:
        raise RuntimeError(
            "Kaggle CLI is not available. Enable INSTALL_KAGGLE_IF_MISSING or install the kaggle package first."
        )

    print("Installing Kaggle CLI with pip...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    verify = subprocess.run(
        [sys.executable, "-m", "kaggle", "--help"],
        capture_output=True,
        text=True,
    )
    if verify.returncode != 0:
        raise RuntimeError("Kaggle CLI installation completed, but the CLI still is not runnable.")


def load_kaggle_api_token_from_secret():
    secret_name = KAGGLE_API_TOKEN_SECRET.strip()
    if not secret_name:
        return ""
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        raise RuntimeError(
            "KAGGLE_API_TOKEN_SECRET was set, but kaggle_secrets is unavailable in this environment."
        ) from exc
    client = UserSecretsClient()
    token = client.get_secret(secret_name)
    return (token or "").strip()


def configure_kaggle_auth():
    token = KAGGLE_API_TOKEN_OVERRIDE.strip()
    if not token:
        token = load_kaggle_api_token_from_secret()
    if not token:
        token = os.environ.get("KAGGLE_API_TOKEN", "").strip()

    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        return "KAGGLE_API_TOKEN"

    access_token_path = Path.home() / ".kaggle" / "access_token"
    if access_token_path.exists():
        return str(access_token_path)

    legacy_path = Path.home() / ".kaggle" / "kaggle.json"
    if legacy_path.exists():
        return str(legacy_path)

    raise RuntimeError(
        "No Kaggle CLI auth found. Set KAGGLE_API_TOKEN_OVERRIDE, set KAGGLE_API_TOKEN_SECRET, "
        "export KAGGLE_API_TOKEN, or place credentials at ~/.kaggle/access_token or ~/.kaggle/kaggle.json."
    )


def probe_dataset_exists():
    result = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "status", DATASET_REF],
        capture_output=True,
        text=True,
    )
    text = f"{result.stdout}\n{result.stderr}".lower()
    if result.returncode == 0:
        return True
    if any(token in text for token in ["404", "not found", "does not exist", "no dataset"]):
        return False
    if any(token in text for token in ["401", "403", "unauthorized", "forbidden", "permission"]):
        raise RuntimeError("Kaggle auth failed while probing dataset status.")
    print("Dataset status probe was inconclusive; defaulting to create mode.")
    return False


def publish_with_kaggle_cli(output_root):
    ensure_kaggle_cli()
    auth_source = configure_kaggle_auth()
    print(f"Kaggle CLI auth: {auth_source}")

    publish_mode = PUBLISH_MODE
    if publish_mode == "auto":
        publish_mode = "version" if probe_dataset_exists() else "create"

    cmd = [
        sys.executable,
        "-m",
        "kaggle",
        "datasets",
        publish_mode,
        "-p",
        str(output_root),
        "--dir-mode",
        "zip",
    ]
    if publish_mode == "create" and PUBLISH_PUBLIC:
        cmd.append("--public")
    if publish_mode == "version":
        message = PUBLISH_MESSAGE.strip() or (
            f"Export {EXPORT_MODE} shard {ACTIVE_SHARD+1}/{NUM_SHARDS} at {TARGET_SIZE}px"
        )
        cmd.extend(["-m", message])

    print(f"Publishing {DATASET_REF} with mode={publish_mode}...")
    run_command(cmd)
    dataset_url = f"https://www.kaggle.com/datasets/{DATASET_REF}"
    print(f"Published dataset: {dataset_url}")
    return {
        "publish_requested": True,
        "published": True,
        "published_mode": publish_mode,
        "dataset_ref": DATASET_REF,
        "dataset_url": dataset_url,
    }


if OUTPUT_ROOT.exists():
    if OVERWRITE_OUTPUT:
        shutil.rmtree(OUTPUT_ROOT)
    else:
        raise RuntimeError(f"Output already exists: {OUTPUT_ROOT}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

write_dataset_metadata(OUTPUT_ROOT)
if EXPORT_MODE == "nabirds":
    write_nabirds_metadata(OUTPUT_ROOT)

manifest_out = active_df.copy()
manifest_out["src_path"] = manifest_out["src_path"].astype(str)
manifest_out.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)

written_bytes = 0
start = time.time()
if not DRY_RUN:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for result in tqdm(executor.map(preprocess_one, task_iter()), total=len(active_df), desc="Preprocessing"):
            written_bytes += int(result)
elapsed = time.time() - start

output_stats = collect_output_stats(OUTPUT_ROOT)

summary = {
    "export_mode": EXPORT_MODE,
    "source_root": str(SOURCE_ROOT),
    "num_shards": NUM_SHARDS,
    "active_shard": ACTIVE_SHARD,
    "items": int(len(active_df)),
    "raw_input_bytes": int(active_raw_bytes),
    "written_output_bytes": int(written_bytes),
    "total_file_bytes": output_stats["total_file_bytes"],
    "image_file_bytes": output_stats["image_file_bytes"],
    "image_file_count": output_stats["image_file_count"],
    "avg_image_kb": output_stats["avg_image_kb"],
    "elapsed_seconds": elapsed,
    "target_size": TARGET_SIZE,
    "jpeg_quality": JPEG_QUALITY,
    "max_workers": MAX_WORKERS,
    "dry_run": DRY_RUN,
    "dataset_ref": DATASET_REF,
    "publish_requested": PUBLISH_TO_KAGGLE,
    "publish_mode": PUBLISH_MODE,
}

if PUBLISH_TO_KAGGLE and DRY_RUN:
    print("Skipping Kaggle publish because DRY_RUN=True.")

if PUBLISH_TO_KAGGLE and not DRY_RUN:
    summary.update(publish_with_kaggle_cli(OUTPUT_ROOT))
else:
    summary.update({
        "published": False,
        "published_mode": None,
        "dataset_url": f"https://www.kaggle.com/datasets/{DATASET_REF}",
    })

(OUTPUT_ROOT / "export_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
write_readme(OUTPUT_ROOT, written_bytes)

print()
print(f"Finished in {elapsed/60:.1f} minutes")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Output written (image bytes counted during preprocessing): {bytes_to_gb(written_bytes):.2f} GB")
print(f"Output on disk (all files): {bytes_to_gb(summary['total_file_bytes']):.2f} GB")
print(f"Image files written: {summary['image_file_count']:,} | avg image size: {summary['avg_image_kb']:.1f} KB")
if active_raw_bytes > 0:
    print(f"Compression ratio vs raw input shard: {summary['image_file_bytes'] / active_raw_bytes:.3f}")
if summary.get("published"):
    print(f"Published: {summary['dataset_url']}")
else:
    print("Publishing not executed. You can enable PUBLISH_TO_KAGGLE or save the output via the Kaggle UI.")
print(f"Dataset ref: {DATASET_REF}")
